In [0]:
#%run ./env ----- A decommenter pour lancer le notebook separement
#%run ./python_libraries ----- A decommenter pour lancer le notebook separement
#%run ../delta_function ----- A decommenter pour lancer le notebook separement
#%run ./load_data ----- A decommenter pour lancer le notebook separement
#%run ./transform_data ----- A decommenter pour lancer le notebook separement

In [0]:
#%run ./env

In [0]:
#%run ./python_libraries 

In [0]:
#%run ../delta_function 

In [0]:
#%run ./load_data 

In [0]:
#%run ./transform_data 

## Construction dim_batch
Une ligne = un batch. Cle primaire = batch_id.

In [0]:
dim_batch = (
    batches
    .filter(F.col("deleted") == False)
    .select(
        F.col("id_batch").alias("batch_id"),
        F.col("batch_number"),
        F.col("requirement_specifications"),
        F.col("variety"),
        F.col("fabrication_order_number"),
        F.col("mes_number"),
        F.col("planned_date"),
        F.col("harvest"),
        F.col("production_line"),
        F.col("production_type"),
        F.col("status"),
        F.col("deleted"),
        F.col("batch_cycle"),
        F.col("planned_time"),
        F.col("planned_datetime"),
        F.col("created_at"),
        F.col("updated_at"),
        F.col("deleted_at")
    )
)

# production_line reste en FK brute (int) : la relation vers dim_site
# (id_plant_production_line) se fait cote Power BI, pas de jointure ici.

# max_end_date_kiln_unload : attribut du batch, calcule une fois, vraie Date.
# Regroupement par batch_id uniquement (pas prd_line) : evite les doublons
# lies a d'eventuelles variations de prd_line pour un meme batch.
max_end_date_kiln_unload = (
    localization_events_union
    .filter(
        (F.col("localization_event") == "kiln_unload") &
        (F.col("batch_id").isNotNull())
    )
    .groupBy("batch_id")
    .agg(F.max("end").alias("max_end_date_kiln_unload_brute"))
    .withColumn("max_end_date_kiln_unload", F.to_date(F.col("max_end_date_kiln_unload_brute")))
    .select("batch_id", "max_end_date_kiln_unload")
)

dim_batch = (
    dim_batch.alias("a")
    .join(
        max_end_date_kiln_unload.alias("b"),
        F.col("a.batch_id") == F.col("b.batch_id"),
        "left"
    )
    .select(
        "a.*",
        "b.max_end_date_kiln_unload"
    )
)

## Resolution des libelles variete et cahier des charges
batches.variety et batches.requirement_specifications sont des FK. On rapatrie ici le libelle depuis goods_varieties et requirement_specifications, pour que le rapport ENERGY V2 affiche et filtre sur des chaines lisibles et non sur des identifiants.

La FK d'origine est conservee sous variety_id / specification_id ; le libelle prend le nom d'origine, ce qui evite de toucher au modele semantique et au rapport.

In [ ]:
# Cle et libelle des tables de reference. CE SONT LES SEULES VALEURS A VERIFIER
# si la source ne suit pas la convention id_<table_au_singulier> / name.
VARIETY_REF_KEY, VARIETY_REF_LABEL = "id_goods_variety", "name"
SPECIF_REF_KEY,  SPECIF_REF_LABEL  = "id_requirement_specification", "name"


def check_colonnes(df, table_name, *colonnes):
    """Echoue tot et avec le detail utile plutot que de produire des libelles null."""
    manquantes = [c for c in colonnes if c not in df.columns]
    if manquantes:
        raise ValueError(
            f"{table_name} : colonne(s) {manquantes} introuvable(s). "
            f"Colonnes disponibles : {df.columns}"
        )


check_colonnes(goods_varieties, "goods_varieties", VARIETY_REF_KEY, VARIETY_REF_LABEL)
check_colonnes(requirement_specifications, "requirement_specifications",
               SPECIF_REF_KEY, SPECIF_REF_LABEL)

# Left join : un batch sans variete ou sans cahier des charges renseigne reste
# dans dim_batch, avec un libelle null. On ne perd aucune ligne.
dim_batch = (
    dim_batch.alias("b")
    .join(
        goods_varieties.select(
            F.col(VARIETY_REF_KEY).alias("_variety_key"),
            F.col(VARIETY_REF_LABEL).alias("_variety_label"),
        ).alias("gv"),
        F.col("b.variety") == F.col("gv._variety_key"),
        "left",
    )
    .join(
        requirement_specifications.select(
            F.col(SPECIF_REF_KEY).alias("_specif_key"),
            F.col(SPECIF_REF_LABEL).alias("_specif_label"),
        ).alias("rs"),
        F.col("b.requirement_specifications") == F.col("rs._specif_key"),
        "left",
    )
    .select("b.*", F.col("gv._variety_label"), F.col("rs._specif_label"))
)

# La FK d'origine devient variety_id / specification_id, le libelle reprend le
# nom d'origine : rien a changer cote modele semantique ni cote rapport, qui
# referencent deja dim_batch[variety] et dim_batch[requirement_specifications].
dim_batch = (
    dim_batch
    .withColumnRenamed("variety", "variety_id")
    .withColumnRenamed("requirement_specifications", "specification_id")
    .withColumnRenamed("_variety_label", "variety")
    .withColumnRenamed("_specif_label", "requirement_specifications")
)

# Controle de qualite non bloquant : une FK renseignee mais non resolue signale
# une reference orpheline cote source. Volontairement un print et non un raise :
# c'est un probleme de donnee, pas de structure, il ne doit pas bloquer le run.
if verbose_mode == 'debug':
    orphelins_variete = dim_batch.filter(
        F.col("variety_id").isNotNull() & F.col("variety").isNull()
    ).count()
    orphelins_specification = dim_batch.filter(
        F.col("specification_id").isNotNull() & F.col("requirement_specifications").isNull()
    ).count()
    if orphelins_variete or orphelins_specification:
        print(
            f"ATTENTION : {orphelins_variete} batch(s) avec une variete non resolue, "
            f"{orphelins_specification} avec un cahier des charges non resolu"
        )


## Ecriture Delta

In [0]:
current_process = "dim_batch"
target_dim_batch = current_catalog + "." + current_schema + "." + current_process
print(target_dim_batch)

In [0]:
all_columns = dim_batch.columns
primary_key = ['batch_id']
additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug':
    display(all_columns)
    print(additional_columns)

In [0]:
handle_table_update(
    dim_batch,
    target_dim_batch,
    primary_key,
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode
)